# Multi-Task Hierarchical DeBERTa for GECS Classification

**Architecture:** DeBERTa-v3-base encoder → 3 classification heads (sector/group/code)

**Key innovations over flat FinBERT:**
- Joint sector + group + code prediction (no cascade error propagation)
- Focal loss with class weights (targets long-tail macro F1)
- Class-balanced sampling (every batch sees rare classes)
- MAX_LEN=512 (captures full descriptions)
- Gradient checkpointing + AMP (fits on T4)

**Runtime:** ~60-90 min on T4 GPU

**Inputs:** `task1_train.csv`, `task1_test.csv`

**Outputs:** predictions, probabilities, embeddings for local ensemble

In [ ]:
# ── 1. Setup ─────────────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), 'GB')

!pip install -q transformers==4.44.2 accelerate==0.33.0 scikit-learn==1.4.2

In [ ]:
# ── 2. Upload CSVs ───────────────────────────────────────────────────────
from google.colab import files
print('Upload task1_train.csv and task1_test.csv:')
uploaded = files.upload()

import pandas as pd
train_df = pd.read_csv('task1_train.csv')
test_df  = pd.read_csv('task1_test.csv')
print(f'train: {len(train_df):,}  test: {len(test_df):,}')
print(f'unique codes: {train_df.mstar_code.nunique()}')

In [ ]:
# ── 3. Data Preparation ──────────────────────────────────────────────────
import numpy as np
from sklearn.preprocessing import LabelEncoder
from collections import Counter

def norm_code(v):
    return str(int(v)).zfill(8)

for df in [train_df, test_df]:
    df['code'] = df['mstar_code'].map(norm_code)
    df['sector'] = df['code'].str[:3]
    df['group'] = df['code'].str[:5]

le_sector = LabelEncoder().fit(train_df['sector'].tolist() + test_df['sector'].tolist())
le_group  = LabelEncoder().fit(train_df['group'].tolist() + test_df['group'].tolist())
le_code   = LabelEncoder().fit(train_df['code'].tolist() + test_df['code'].tolist())

for df in [train_df, test_df]:
    df['sector_idx'] = le_sector.transform(df['sector'])
    df['group_idx']  = le_group.transform(df['group'])
    df['code_idx']   = le_code.transform(df['code'])

N_SECTORS = len(le_sector.classes_)
N_GROUPS  = len(le_group.classes_)
N_CODES   = len(le_code.classes_)
print(f'Sectors: {N_SECTORS}  Groups: {N_GROUPS}  Codes: {N_CODES}')

# Show class distribution
code_counts = Counter(train_df['code'].tolist())
counts = sorted(code_counts.values())
print(f'Code support: min={counts[0]} median={counts[len(counts)//2]} max={counts[-1]}')

In [ ]:
# ── 4. Tokenize ──────────────────────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LEN = 512

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Tokenizing train...')
tr_enc = tok(train_df['text'].tolist(), padding='max_length', truncation=True,
             max_length=MAX_LEN, return_tensors='pt')
print('Tokenizing test...')
te_enc = tok(test_df['text'].tolist(), padding='max_length', truncation=True,
             max_length=MAX_LEN, return_tensors='pt')
print(f'Train tokens: {tr_enc["input_ids"].shape}  Test tokens: {te_enc["input_ids"].shape}')

In [ ]:
# ── 5. Model Architecture ────────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class FocalLoss(nn.Module):
    """Focal loss: downweights easy examples, focuses on hard ones."""
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             reduction='none', label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


class MultiTaskHTC(nn.Module):
    """DeBERTa encoder with 3 classification heads for hierarchical prediction."""
    def __init__(self, model_name, n_sectors, n_groups, n_codes, dropout=0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.gradient_checkpointing_enable()
        h = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.sector_head = nn.Linear(h, n_sectors)
        self.group_head  = nn.Linear(h, n_groups)
        self.code_head   = nn.Linear(h, n_codes)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.drop(out.last_hidden_state[:, 0, :])
        return self.sector_head(cls), self.group_head(cls), self.code_head(cls), cls

model = MultiTaskHTC(MODEL_NAME, N_SECTORS, N_GROUPS, N_CODES)
device = torch.device('cuda')
model = model.to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 6. Training Setup ────────────────────────────────────────────────────
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight

EPOCHS = 5
BATCH  = 8
GRAD_ACCUM = 8  # effective batch = 64
LR = 2e-5

# Class weights for each level
def make_weights(labels, n_classes):
    w = compute_class_weight('balanced', classes=np.arange(n_classes), y=labels)
    return torch.tensor(w, dtype=torch.float32, device=device)

w_sector = make_weights(train_df['sector_idx'].values, N_SECTORS)
w_group  = make_weights(train_df['group_idx'].values, N_GROUPS)
w_code   = make_weights(train_df['code_idx'].values, N_CODES)

loss_sector = FocalLoss(weight=w_sector, gamma=2.0, label_smoothing=0.05)
loss_group  = FocalLoss(weight=w_group,  gamma=2.0, label_smoothing=0.05)
loss_code   = FocalLoss(weight=w_code,   gamma=2.0, label_smoothing=0.05)

# Multi-task loss weights
W_SECTOR, W_GROUP, W_CODE = 0.15, 0.15, 0.70

# Datasets
tr_ds = TensorDataset(
    tr_enc['input_ids'], tr_enc['attention_mask'],
    torch.tensor(train_df['sector_idx'].values),
    torch.tensor(train_df['group_idx'].values),
    torch.tensor(train_df['code_idx'].values),
)
te_ds = TensorDataset(
    te_enc['input_ids'], te_enc['attention_mask'],
    torch.tensor(test_df['sector_idx'].values),
    torch.tensor(test_df['group_idx'].values),
    torch.tensor(test_df['code_idx'].values),
)

# Class-balanced sampler: each code gets ~equal representation
code_counts_arr = Counter(train_df['code_idx'].tolist())
sample_weights = torch.tensor([1.0 / code_counts_arr[c] for c in train_df['code_idx'].values])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

tr_loader = DataLoader(tr_ds, batch_size=BATCH, sampler=sampler,
                       num_workers=2, pin_memory=True, drop_last=True)
te_loader = DataLoader(te_ds, batch_size=BATCH * 2, shuffle=False,
                       num_workers=2, pin_memory=True)

# Optimizer + scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
n_steps = (len(tr_loader) // GRAD_ACCUM) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer,
    num_warmup_steps=int(n_steps * 0.1), num_training_steps=n_steps)
scaler = torch.cuda.amp.GradScaler()

print(f'Steps/epoch: {len(tr_loader)}  Total optim steps: {n_steps}')
print(f'Effective batch: {BATCH * GRAD_ACCUM}')

In [ ]:
# ── 7. Training Loop ─────────────────────────────────────────────────────
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score
import time, os, json

os.makedirs('htc_outputs', exist_ok=True)
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    t0 = time.time()

    for step, (ids, mask, s_lbl, g_lbl, c_lbl) in enumerate(tqdm(tr_loader, desc=f'epoch {epoch+1}/{EPOCHS}')):
        ids, mask = ids.to(device), mask.to(device)
        s_lbl, g_lbl, c_lbl = s_lbl.to(device), g_lbl.to(device), c_lbl.to(device)

        with torch.cuda.amp.autocast():
            s_logits, g_logits, c_logits, _ = model(ids, mask)
            loss = (W_SECTOR * loss_sector(s_logits, s_lbl) +
                    W_GROUP  * loss_group(g_logits, g_lbl) +
                    W_CODE   * loss_code(c_logits, c_lbl))
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

    avg_loss = total_loss / len(tr_loader)
    elapsed = time.time() - t0

    # ── Evaluate ──
    model.eval()
    all_preds, all_true, all_s_pred, all_s_true = [], [], [], []
    with torch.no_grad():
        for ids, mask, s_lbl, g_lbl, c_lbl in tqdm(te_loader, desc='eval'):
            ids, mask = ids.to(device), mask.to(device)
            with torch.cuda.amp.autocast():
                s_logits, g_logits, c_logits, _ = model(ids, mask)
            all_preds.extend(c_logits.argmax(dim=-1).cpu().tolist())
            all_true.extend(c_lbl.tolist())
            all_s_pred.extend(s_logits.argmax(dim=-1).cpu().tolist())
            all_s_true.extend(s_lbl.tolist())

    pred_codes = le_code.inverse_transform(all_preds)
    true_codes = le_code.inverse_transform(all_true)
    macro_f1 = f1_score(true_codes, pred_codes, average='macro', zero_division=0)
    acc = accuracy_score(true_codes, pred_codes)
    s_acc = accuracy_score(all_s_true, all_s_pred)

    print(f'\nepoch {epoch+1}: loss={avg_loss:.4f}  time={elapsed:.0f}s')
    print(f'  Sector accuracy : {s_acc*100:.2f}%')
    print(f'  Code Macro F1   : {macro_f1*100:.2f}%')
    print(f'  Code Accuracy   : {acc*100:.2f}%')

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), 'htc_outputs/best_model.pt')
        print(f'  ★ New best! Saved.')

print(f'\n{"="*60}')
print(f'BEST MACRO F1: {best_f1*100:.2f}%')
print(f'{"="*60}')

In [ ]:
# ── 8. Final Evaluation with Best Model ──────────────────────────────────
model.load_state_dict(torch.load('htc_outputs/best_model.pt'))
model.eval()

all_preds, all_probs, all_true = [], [], []
all_embeds = []

with torch.no_grad():
    for ids, mask, s_lbl, g_lbl, c_lbl in tqdm(te_loader, desc='final eval'):
        ids, mask = ids.to(device), mask.to(device)
        with torch.cuda.amp.autocast():
            _, _, c_logits, cls_emb = model(ids, mask)
        probs = torch.softmax(c_logits.float(), dim=-1).cpu().numpy()
        all_preds.extend(c_logits.argmax(dim=-1).cpu().tolist())
        all_probs.append(probs)
        all_true.extend(c_lbl.tolist())
        all_embeds.append(cls_emb.float().cpu().numpy())

all_probs = np.vstack(all_probs)
all_embeds_te = np.vstack(all_embeds)
pred_codes = le_code.inverse_transform(all_preds)
true_codes = le_code.inverse_transform(all_true)

macro_f1 = f1_score(true_codes, pred_codes, average='macro', zero_division=0)
acc = accuracy_score(true_codes, pred_codes)

# Top-10 breakdown
cf = Counter(true_codes.tolist())
top10 = [c for c, _ in cf.most_common(10)]
f1s = f1_score(true_codes, pred_codes, average=None, labels=top10, zero_division=0)
n_pass = int(sum(1 for v in f1s if v > 0.85))

# Tail class F1 (codes with <= 50 test samples)
tail_codes = [c for c, n in cf.items() if n <= 50]
if tail_codes:
    tail_f1 = f1_score(true_codes, pred_codes, average='macro', labels=tail_codes, zero_division=0)
else:
    tail_f1 = 0.0

print(f'\n{"="*60}')
print(f'MULTI-TASK HIERARCHICAL DeBERTa — FINAL RESULT')
print(f'{"="*60}')
print(f'Macro F1   : {macro_f1*100:.2f}%')
print(f'Accuracy   : {acc*100:.2f}%')
print(f'Tail F1    : {tail_f1*100:.2f}% ({len(tail_codes)} codes with ≤50 samples)')
print(f'Top-10 pass: {n_pass}/10')
for c, v in zip(top10, f1s):
    flag = 'PASS' if v > 0.85 else 'FAIL'
    print(f'  [{flag}] {c}: F1={v*100:.1f}%  n={cf[c]}')

print(f'\nTarget >=75%: {"PASS" if macro_f1 >= 0.75 else "FAIL"}')
print(f'Target >=80%: {"PASS" if macro_f1 >= 0.80 else "FAIL"}')

In [ ]:
# ── 9. Extract Train Embeddings ──────────────────────────────────────────
print('Extracting train embeddings for ensemble stacking...')
tr_loader_seq = DataLoader(tr_ds, batch_size=BATCH * 2, shuffle=False,
                           num_workers=2, pin_memory=True)
all_embeds_tr = []
all_probs_tr  = []

with torch.no_grad():
    for ids, mask, s_lbl, g_lbl, c_lbl in tqdm(tr_loader_seq, desc='train embed'):
        ids, mask = ids.to(device), mask.to(device)
        with torch.cuda.amp.autocast():
            _, _, c_logits, cls_emb = model(ids, mask)
        all_embeds_tr.append(cls_emb.float().cpu().numpy())
        all_probs_tr.append(torch.softmax(c_logits.float(), dim=-1).cpu().numpy())

all_embeds_tr = np.vstack(all_embeds_tr)
all_probs_tr  = np.vstack(all_probs_tr)
print(f'Train embeddings: {all_embeds_tr.shape}  Test embeddings: {all_embeds_te.shape}')

In [ ]:
# ── 10. Save & Download ──────────────────────────────────────────────────

# Predictions CSV
pd.DataFrame({
    'true_code': true_codes,
    'pred_code': pred_codes,
}).to_csv('htc_outputs/htc_test_predictions.csv', index=False)

# Probability matrices
np.save('htc_outputs/htc_test_probs.npy', all_probs)
np.save('htc_outputs/htc_train_probs.npy', all_probs_tr)

# Embeddings for stacking
np.save('htc_outputs/htc_embeddings_train.npy', all_embeds_tr)
np.save('htc_outputs/htc_embeddings_test.npy', all_embeds_te)

# Label encoder classes
np.save('htc_outputs/le_code_classes.npy', le_code.classes_)
np.save('htc_outputs/le_sector_classes.npy', le_sector.classes_)
np.save('htc_outputs/le_group_classes.npy', le_group.classes_)

# Summary
summary = {
    'model': MODEL_NAME,
    'architecture': 'multi-task-htc',
    'heads': {'sector': N_SECTORS, 'group': N_GROUPS, 'code': N_CODES},
    'loss': 'focal (gamma=2.0, label_smoothing=0.05)',
    'loss_weights': {'sector': W_SECTOR, 'group': W_GROUP, 'code': W_CODE},
    'max_len': MAX_LEN,
    'epochs': EPOCHS,
    'batch': BATCH,
    'grad_accum': GRAD_ACCUM,
    'lr': LR,
    'macro_f1': round(float(macro_f1) * 100, 2),
    'accuracy': round(float(acc) * 100, 2),
    'tail_f1': round(float(tail_f1) * 100, 2),
    'top10_pass': n_pass,
    'target_75_met': bool(macro_f1 >= 0.75),
    'target_80_met': bool(macro_f1 >= 0.80),
}
with open('htc_outputs/htc_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

# Zip and download
!zip -qr htc_outputs.zip htc_outputs
files.download('htc_outputs.zip')
print('\nDownloaded htc_outputs.zip — unzip into your project as models_v18/htc_outputs/')